We said in our second notebook that we would have compared the pre-tokenized data kindly given to us by the CodeXGLUE database, with a BPE tokenizer. That is because, as explained in this [paper](https://www.researchgate.net/publication/394590942_How_Different_Tokenization_Algorithms_Impact_LLMs_and_Transformer_Models_for_Binary_Code_Analysis) published in 2025, that seems to be the best combination with an LSTM for code summarization. 

We are gonna try to use the [Huggin Face tokenizers](https://github.com/huggingface/tokenizers).

In [19]:
from datasets import load_dataset
train_dataset = load_dataset(
  "json", 
  data_files="./../../data/raw/dataset/python/train.jsonl",
  split="train"
)
valid_dataset = load_dataset(
  "json",
  data_files="./../../data/raw/dataset/python/valid.jsonl",
  split="train"
)
test_dataset = load_dataset(
  "json",
  data_files="./../../data/raw/dataset/python/test.jsonl",
  split="train"
)

Generating train split: 13914 examples [00:00, 113665.36 examples/s]
Generating train split: 14918 examples [00:00, 108194.90 examples/s]


Now, if you remember there's a little difference between 'docstring_tokens' and 'docstring'. Let's see it:

In [12]:
print(train_dataset['docstring_tokens'][0])
print(train_dataset['docstring'][0])

['Return', 'either', 'the', 'full', 'or', 'truncated', 'version', 'of', 'a', 'QIIME', '-', 'formatted', 'taxonomy', 'string', '.']
Return either the full or truncated version of a QIIME-formatted taxonomy string.

    :type p: str
    :param p: A QIIME-formatted taxonomy string: k__Foo; p__Bar; ...

    :type level: str
    :param level: The different level of identification are kingdom (k), phylum (p),
                  class (c),order (o), family (f), genus (g) and species (s). If level is
                  not provided, the default level of identification is species.

    :rtype: str
    :return: A QIIME-formatted taxonomy string up to the classification given
            by param level.


We can see that 'docstring' is much more complex, as it explaines every parameter. We don't want that for 2 main reasons: first of all, we trained our last models with 'docstring_tokens', so it would be unfair to train this differently. Secondly, our vocab would explode, probably tanking our performances. So we're gonna use 'docstring_tokens', as it's just the simple docstring splitted on spaces, and rejoin it, so we can feed it to the tokenizer as if it was the original.

In [13]:
print(" ".join(train_dataset['docstring_tokens'][1]))

Check to make sure the supplied directory path does not exist if so create it . The method catches OSError exceptions and returns a descriptive message instead of re - raising the error .


In [18]:
import os
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFD, Lowercase, StripAccents, Sequence

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.normalizer = Sequence([NFD(), Lowercase(), StripAccents()])

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
  vocab_size = 8000,
  min_frequency = 2,
  special_tokens = ["[UNK]", "[EOS]", "[PAD]", "[BOS]"]
)

tokenizer.train_from_iterator([" ".join(tokens) for tokens in train_dataset['docstring_tokens']], trainer=trainer)

save_path = "./../../data/processed/ast_BPE/bpe_tokenizer.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

tokenizer.save(save_path)

let's now tokenize the data with our tokenizer and save it.

In [32]:


train_encoded = [tokenizer.encode(" ".join(docstring)).ids for docstring in train_dataset['docstring_tokens']]
valid_encoded = [tokenizer.encode(" ".join(docstring)).ids for docstring in valid_dataset['docstring_tokens']]
test_encoded = [tokenizer.encode(" ".join(docstring)).ids for docstring in test_dataset['docstring_tokens']]

In [34]:
print(train_encoded[0])

[97, 996, 65, 1021, 69, 4691, 433, 86, 32, 48, 40, 926, 10, 1332, 7162, 203, 11]


In [35]:
from datasets import Dataset, DatasetDict
from datasets import load_from_disk
old_tokenizer = load_from_disk("../../data/processed/ast01/")

processed_datasets = DatasetDict({
  'train': Dataset.from_dict({
    'input_ids': old_tokenizer['train']['input_ids'],
    'labels': train_encoded
  }),
  'valid': Dataset.from_dict({
    'input_ids': old_tokenizer['valid']['input_ids'],
    'labels': valid_encoded
  }),
  'test': Dataset.from_dict({
    'input_ids': old_tokenizer['test']['input_ids'],
    'labels': test_encoded
  })
})

In [ ]:
from gensim import corpora

ast_dictionary = corpora.Dictionary.load('./../../data/processed/ast01/code_dictionary.pt')

processed_datasets.save_to_disk('./../../data/processed/ast_BPE/')
ast_dictionary.save('./../../data/processed/ast_BPE/code_dictionary.pt')